# Response generation

[Fine-tuning Llama 2 with LoRA](https://deci.ai/blog/fine-tune-llama-2-with-lora-for-question-answering/)

# Testing Data
testing different data structure

In [1]:
import os, torch, logging
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, HfArgumentParser, TrainingArguments, pipeline
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

from flash_attn import flash_attn_func
import accelerate

import wandb
from huggingface_hub import login
import os

In [2]:
%load_ext dotenv
%dotenv

login(token=os.environ.get("HF_TOKEN", ""), add_to_git_credential=True)

Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (cache).
Your token has been saved to /home/user/.cache/huggingface/token
Login successful


### Dataset

Observe first 10% of data from better_daily_dialog

In [3]:
dd = load_dataset("benjaminbeilharz/better_daily_dialog", split="train[:10%]")

In [4]:
dd[0:10]['utterance']

['Say , Jim , how about going for a few beers after dinner ? ',
 ' You know that is tempting but is really not good for our fitness . ',
 ' What do you mean ? It will help us to relax . ',
 " Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ? ",
 " I guess you are right.But what shall we do ? I don't feel like sitting at home . ",
 ' I suggest a walk over to the gym where we can play singsong and meet some of our friends . ',
 " That's a good idea . I hear Mary and Sally often go there to play pingpong.Perhaps we can make a foursome with them . ",
 ' Sounds great to me ! If they are willing , we could ask them to go dancing with us.That is excellent exercise and fun , too . ',
 " Good.Let ' s go now . ",
 ' All right . ']

### Model
setup model

In [5]:
# Model and tokenizer names
base_model_name = "meta-llama/Llama-2-7b-chat-hf"
refined_model = "llama-2-7b-testing"

# Tokenizer
llama_tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
llama_tokenizer.pad_token = llama_tokenizer.eos_token
llama_tokenizer.padding_side = "right"  # Fix for fp16

# Quantization Config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False
)

# Model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    attn_implementation="flash_attention_2",
    quantization_config=quant_config,
    device_map={"": 0},
    use_cache=False,
    low_cpu_mem_usage=True
)

base_model.config.use_cache = False
base_model.config.pretraining_tp = 1

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

### Data
paking up and shits...

In [16]:
# I don't think this is a good idea

chat1 = [
    {"role": "system", "content": "rewrite with anger emotions"},
    {"role": "user", "content": "Say , Jim , how about going for a few beers after dinner ? "},
    {"role": "assistant", "content": "Jim, are you serious? We're talking about going for beers after dinner?"}
]
chat2 = [
    {"role": "system", "content": "rewrite with anger emotions"},
    {"role": "user", "content": "You know that is tempting but is really not good for our fitness . "},
    {"role": "assistant", "content": "You're aware how tempting that is, but it's seriously detrimental to our fitness."}
]
chat3 = [
    {"role": "system", "content": "rewrite with anger emotions"},
    {"role": "user", "content": "What do you mean ? It will help us to relax . "},
    {"role": "assistant", "content": "What do I mean? Are you serious? You think that's going to help us relax?"}
]
chat4 = [
    {"role": "system", "content": "rewrite with anger emotions"},
    {"role": "user", "content": "Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ? "},
    {"role": "assistant", "content": "Do I really think so? Of course not! It's just going to make us fat and act stupid. Remember the last time?"}
]
chat5 = [
    {"role": "system", "content": "rewrite with anger emotions"},
    {"role": "user", "content": "I guess you are right.But what shall we do ? I don't feel like sitting at home . "},
    {"role": "assistant", "content": "You guess I'm right? What's that supposed to mean? Fine, if you don't want to sit at home, figure something else out."}
]

dataset = Dataset.from_dict({"chat": [chat1, chat2, chat3, chat4, chat5]})
dataset = dataset.map(lambda x: {"formatted_chat": llama_tokenizer.apply_chat_template(x["chat"], tokenize=False, add_generation_prompt=False)})
print(dataset['formatted_chat'])

test_dataset = Dataset.from_dict({"chat": [chat1[:2], chat2[:2], chat3[:2], chat4[:2], chat5[:2]]})
test_dataset = test_dataset.map(lambda x: {"formatted_chat": llama_tokenizer.apply_chat_template(x["chat"], tokenize=False, add_generation_prompt=False)})
print(test_dataset['formatted_chat'])

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

["<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\nSay , Jim , how about going for a few beers after dinner ? [/INST] Jim, are you serious? We're talking about going for beers after dinner? </s>", "<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\nYou know that is tempting but is really not good for our fitness . [/INST] You're aware how tempting that is, but it's seriously detrimental to our fitness. </s>", "<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\nWhat do you mean ? It will help us to relax . [/INST] What do I mean? Are you serious? You think that's going to help us relax? </s>", "<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\nDo you really think so ? I don't . It will just make us fat and act silly . Remember last time ? [/INST] Do I really think so? Of course not! It's just going to make us fat and act stupid. Remember the last time? </s>", "<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\nI guess you are right.But wha

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

['<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\nSay , Jim , how about going for a few beers after dinner ? [/INST]', '<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\nYou know that is tempting but is really not good for our fitness . [/INST]', '<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\nWhat do you mean ? It will help us to relax . [/INST]', "<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\nDo you really think so ? I don't . It will just make us fat and act silly . Remember last time ? [/INST]", "<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\nI guess you are right.But what shall we do ? I don't feel like sitting at home . [/INST]"]


### Saving them for later use
我不太確定能不能用 `train_test_split` 直接切一切，姑且先分開存反正很小。

In [31]:
dataset.to_json("dataset_v1.json")
test_dataset.to_json("test_dataset_v1.json")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

1635

### Testing Dataset on OG llama 2

In [19]:
pipe = pipeline(task="text-generation", model=base_model, tokenizer=llama_tokenizer)

generated_texts = []

for example in test_dataset:
    chat_text = example['chat']
    generated_text = pipe(chat_text, max_new_tokens=200)[0]['generated_text']
    generated_texts.append(generated_text)
    
for i in range(5):
    print(generated_texts[i][1]['content'])
    print(generated_texts[i][2]['content'])
    print('-' * 100)